# 🧠 Meningioma Modelling Notebook

Run from the **repo root** (`meningioma-atypier/`). Consumes `output/datasets/` from the cleaning notebook.

DDA + EDA on **unimputed** data. Multivariable modelling on **imputed** data.

<details>
<summary><b>Pipeline map</b> — notebook step → module</summary>

| Step | What it does | Driven by |
|------|--------------|-----------|
| 00 | Setup — imports, keep `output/` | `config/` loader + phase modules |
| 01 | Load handoff (data + schema) | `dataset_handoff.load_modelling_handoff` |
| 02 | EDA / model variant lists | `config/07_analysis.py` |
| 03 | DDA on unimputed cohort | `dda` · `run_dda` |
| 04 | EDA + diagnostic accuracy | `eda` · `diagnostic_accuracy` |
| 06 | Multivariable logistic (Rubin pool) | `inferential` → `output/inferential/` (incl. `model_artifacts/`) |
| 07 | HTML report | `config/08_report_settings.py` · `report` |

Config steps 01–08 live in `config/` (loaded via `load("NN_name")`).

</details>


## 00. Setup

⚙️ Loads modelling modules. Does **not** wipe `output/` — reads cleaning handoff artifacts.

<details>
<summary>🔧 How it works</summary>

- 📦 `from heavy_machinery.config import load` plus `heavy_machinery.cleaning_phase.*` and `heavy_machinery.modelling_phase.*` imports.
- 📁 `OUTPUT_ROOT = Path("output")` — same tree as the cleaning notebook.

</details>


In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)
import pandera.pandas as pa

from pathlib import Path

from IPython.display import display

from heavy_machinery.config import load
from heavy_machinery.cleaning_phase.cleaning import format_table_for_display
from heavy_machinery.cleaning_phase.dda import run_dda
from heavy_machinery.cleaning_phase.dataset_handoff import load_modelling_handoff
from heavy_machinery.cleaning_phase.missingness_resolution import load_modeling_frames
from heavy_machinery.cleaning_phase.validation import load_schema_validation, validate_imputed_frames
from heavy_machinery.modelling_phase.eda import screen_associations
from heavy_machinery.modelling_phase.diagnostic_accuracy import screen_diagnostic_accuracy
from heavy_machinery.modelling_phase.inferential import run_inferential_stage, preview_multivariable_cases

OUTPUT_ROOT = Path("output")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)  # do not wipe — reads cleaning outputs

#🟧🟧🟧 None = all years; e.g. [2025] for one cohort year

ANALYSIS_YEARS: list[int] | None = None


## 01. Load handoff

Requires `output/datasets/` parquets and `output/schema/schema_summary.csv` from cleaning §16.

`load_modelling_handoff` loads the **unimputed** cohort (`df`) for DDA/EDA, plus the ColSpec schema and which imputation method was run.

Next cell loads `output/cleaning/schema_validation.json` and validates the unimputed handoff only. Imputed draws are validated again in §06, immediately before multivariable modelling.


In [ ]:
df, schema, IMPUTATION_METHOD = load_modelling_handoff(OUTPUT_ROOT)
print(f"Loaded unimputed cohort: {len(df)} rows, {len(schema)} schema columns")
if IMPUTATION_METHOD == "mice":
    print("Imputation method: MICE — inferential stage (§06) pools multiple imputed draws.")
else:
    print("Imputation method: simple — inferential stage (§06) uses one imputed parquet.")
df.head(3)


In [ ]:
schema_validation = load_schema_validation(OUTPUT_ROOT / "cleaning" / "schema_validation.json")
schema_validation(df, lazy=True)
print("✅ Pandera validated unimputed handoff (DDA/EDA cohort)")

In [ ]:
# 🟧🟧🟧 Copy-pasteable column names from the loaded cohort
load("07_analysis").print_copy_pasteable_columns(df)

### 🎯 EDA


In [ ]:
EDA_TARGETS = ['high_grade', 'progesterone_pos', 'brain_invasion', 'ki67_group', 'hist_necrosis']
# Binary targets only: {column: value coded as positive (1)}.
# Omit → auto-detect: True, else 1, else last sorted level (string binaries).
EDA_POSITIVE_CLASS = {
    'progesterone_pos': True,
    'brain_invasion': True,
    'hist_necrosis': True,
}
EDA_PREDICTORS = [
    'entry_year',
    'age',
    'sex',
    'who_grade',
    'progesterone_pos',
    'brain_invasion',
    'hist_necrosis',
    'side',
    'tumor_location',
    'meningioma_count',
    'max_diameter_cm',
    'tumor_volume',
    'tumor_episode',
    'tumor_margin',
    'dural_tail',
    'capsular_enhancement',
    'heterogeneous_enhancement',
    'perifocal_edema',
    'edema_volume_cm3',
    'mass_effect',
    'calcification',
    'cystic_component',
    'mri_necrosis',
    'hemorrhage',
    'hyperostosis',
    'cortical_destruction',
    'dwi_hyperintensity',
    't2_hyperintensity',
    't1_hypointensity',
    'sinus_invasion',
    'transfalcine_extension',
    'adc_value',
    'age_bins_10',
    'high_grade',
    'multiple_meningiomas',
    'ki67_mid',
    'ki67_group',
    ]

### 📚 Literature-based multivariable models


In [ ]:
# 📚 Literature-based multivariable models — published predictor sets.
# Each variant gets its own EPV bar, forest plot, VIF table, and interpretation.
# Format: (id, title, link, target, [predictors])
LITERATURE_MODEL_VARIANTS = [
    # Research work: Predicting the grade of meningiomas by clinical–radiological features: A comparison of precontrast and postcontrast MRI
    # Authors: Yuan Yao, Yifan Xu, Shihe Liu, Feng Xue, Bao Wang, Shanshan Qin, Xiubin Sun, Jingzhen He
    # Link: https://www.frontiersin.org/journals/oncology/articles/10.3389/fonc.2022.1053089/full
    (
        "yao_et_al_2022",
        "Yao et al. 2022 | precontrast / semantic MRI model",
        "https://www.frontiersin.org/journals/oncology/articles/10.3389/fonc.2022.1053089/full",
        "high_grade",
        [
            "sex",
            "tumor_margin",
            "cystic_component",
            "perifocal_edema",
            "dural_tail",
        ],
    ),

    # Research work: Preoperative Prediction of Intracranial Meningioma Grade Using Conventional CT and MRI
    # Authors: T. Amano et al.
    # Link: https://www.cureus.com/articles/80763-preoperative-prediction-of-intracranial-meningioma-grade-using-conventional-ct-and-mri
    (
        "amano_et_al_2021_expanded_proxy",
        "Amano et al. 2021 expanded proxy | conventional CT/MRI + tumor burden",
        "https://www.cureus.com/articles/80763-preoperative-prediction-of-intracranial-meningioma-grade-using-conventional-ct-and-mri",
        "high_grade",
        [
            "tumor_location",
            "tumor_margin",
            "heterogeneous_enhancement",
            "perifocal_edema",
            "tumor_volume",
            "cortical_destruction",
        ],
    ),

    # Research work: The Role of Pre-Operative MRI for Prediction of High-Grade Intracranial Meningioma: A Retrospective Study
    # Authors: Kan Radeesri, Vitit Lekhavat
    # Link: https://journal.waocp.org/article_90552.html
    (
        "radeesri_lekhavat_2020",
        "Radeesri & Lekhavat 2020 | edema / necrosis MRI model",
        "https://journal.waocp.org/article_90552.html",
        "high_grade",
        [
            "perifocal_edema",
            "edema_volume_cm3",
            "mri_necrosis",
            "hemorrhage",
            "hyperostosis",
            "mass_effect",
        ],
    ),

    # Research work: Role of ADC values and ratios of MRI scan in differentiating typical, atypical and anaplastic meningiomas
    # Authors: M. Azeemuddin et al.
    # Link: https://pubmed.ncbi.nlm.nih.gov/30317276/
    (
        "azeemuddin_et_al_2018",
        "Azeemuddin et al. 2018 | diffusion-augmented MRI model",
        "https://pubmed.ncbi.nlm.nih.gov/30317276/",
        "high_grade",
        [
            "adc_value",
            "dwi_hyperintensity",
            "tumor_location",
            "tumor_margin",
            "perifocal_edema",
            "heterogeneous_enhancement",
            "sex",
        ],
    ),

    # Research work: Diagnostic nomogram model for predicting preoperative pathological grade of meningioma
    # Authors: Shijun Peng, Zhihua Cheng, Zhilin Guo
    # Link: https://tcr.amegroups.org/article/view/55552/html
    (
        "peng_cheng_guo_2021",
        "Peng, Cheng & Guo 2021 | interface / invasion model",
        "https://tcr.amegroups.org/article/view/55552/html",
        "high_grade",
        [
            "tumor_location",
            "tumor_margin",
            "sinus_invasion",
            "cortical_destruction",
            "mass_effect",
            "edema_volume_cm3",
            "hyperostosis",
        ],
    ),
]


### 🧪 Experimental multivariable models


In [ ]:
# 🧪 Experimental multivariable models — your own predictor sets (independent of EDA_PREDICTORS).
# Add as many as you need. Each row is one model: (id, title, link, target, [predictors]).
# Grouping in the report follows this list, not the model id string.
EXPERIMENTAL_MODEL_VARIANTS = [
    (
        "experimental_model_1",
        "model 1 | high grade",
        "",
        "high_grade",
        [
            'cystic_component',
            'cortical_destruction',
            'dural_tail',
            'tumor_volume',
            'edema_volume_cm3',
            'hyperostosis',
            'mass_effect',
            'adc_value',
            'tumor_margin',
        ],
    ),
    (
        "experimental_model_2",
        "model 2 | high grade",
        "",
        "high_grade",
        [
            'dwi_hyperintensity',
            'sex',
            'heterogeneous_enhancement',
            'hemorrhage',
            'sinus_invasion',
            'age_bins_10',
            'calcification',
            't2_hyperintensity',
            't1_hypointensity',
            'transfalcine_extension',

        ],
    ),
]


In [ ]:
_c07 = load("07_analysis")

EDA_TARGETS, EDA_PREDICTORS = _c07.resolve_eda(df, EDA_TARGETS, EDA_PREDICTORS)

INFERENTIAL_MODEL_VARIANTS = _c07.resolve_inferential_variants(
    df,
    LITERATURE_MODEL_VARIANTS,
    EXPERIMENTAL_MODEL_VARIANTS,
)
INFERENTIAL_TARGETS = _c07.resolve_inferential_targets(df, INFERENTIAL_MODEL_VARIANTS)
# Binary inferential targets only: {column: value coded as positive (1)}. Omit → auto-detect.
INFERENTIAL_POSITIVE_CLASS = {}


## 04. DDA on unimputed data

`dda.run_dda` on the unimputed handoff cohort.


In [ ]:
dda_tables = run_dda(df, schema, output_root=OUTPUT_ROOT)

for name in ("overall", "continuous", "categorical", "binary", "datetime", "id_text"):
    print(f"\n--- {name} ---")
    tbl = dda_tables.get(name)
    if tbl is None or tbl.empty:
        print("(none)")
    else:
        display(format_table_for_display(tbl))

## 05. EDA on unimputed data

`eda.screen_associations` and `diagnostic_accuracy.screen_diagnostic_accuracy`.



Targets can be **binary**, **continuous**, **ordinal**, or **nominal** (from schema). The test depends on both outcome and predictor types — e.g. ordinal outcome × nominal predictor → χ²; continuous outcome × nominal predictor → Kruskal–Wallis.

| target kind   | continuous / count predictor | ordinal predictor | nominal / binary predictor |
|---------------|------------------------------|-------------------|----------------------------|
| binary        | Mann–Whitney U               | Spearman ρ        | χ² / Fisher                |
| continuous    | Spearman ρ                   | Spearman ρ        | Kruskal–Wallis             |
| ordinal       | Spearman ρ                   | Spearman ρ        | χ²                         |
| nominal       | Kruskal–Wallis               | χ²                | χ²                         |

`POSITIVE_CLASS` applies only to **binary** targets. Multivariable logistic (§16) remains **binary outcomes only**.

Per-target p-values are corrected with **Benjamini–Hochberg FDR**.


In [ ]:
assoc = screen_associations(
    df, schema,
    targets=EDA_TARGETS,
    predictors=EDA_PREDICTORS,
    positive_class=EDA_POSITIVE_CLASS,
    fdr_alpha=0.05,
    output_root=OUTPUT_ROOT,
    )

diag_acc = screen_diagnostic_accuracy(
    df, schema,
    targets=EDA_TARGETS,
    predictors=EDA_PREDICTORS,
    positive_class=EDA_POSITIVE_CLASS,
    fdr_alpha=0.05,
    output_root=OUTPUT_ROOT,
)

#assoc[assoc['fdr_significant']]

In [ ]:
#🟧🟧🟧 Full table
#assoc

## 06. Multivariable modelling on imputed data

Load imputed dataset(s), Pandera-validate immediately before fitting, then `inferential.run_inferential_stage` (Rubin-pooled across MICE draws).

Writes per-variant tables and forest plots under `output/inferential/`, plus Streamlit JSON under `output/inferential/model_artifacts/`. Re-running clears stale variant files first.


In [ ]:
imputed_frames = load_modeling_frames(OUTPUT_ROOT)
validate_imputed_frames(schema_validation, imputed_frames)

In [ ]:
full_inferential_table = run_inferential_stage(
    schema,
    imputed_frames=imputed_frames,
    targets=INFERENTIAL_TARGETS,
    variants=INFERENTIAL_MODEL_VARIANTS,
    positive_class=INFERENTIAL_POSITIVE_CLASS,
    output_root=OUTPUT_ROOT,
    )
#full_inferential_table

## 07. Build report.html

`config/08_report_settings.py` + `report`. Assembles a self-contained HTML report from artifacts already in `output/` (DDA, EDA, inferential). Launch the calculator separately: `streamlit run app.py` (reads `output/inferential/model_artifacts/`).



Builds `report.html` from artifacts already in `output/` (DDA, EDA, inferential).
Edit the settings cell, then run both cells.

- **`REPORT_TITLE` / `REPORT_AUTHOR`** — shown on the cover.
- **`REPORT_PATH`** — where to write the HTML file.
- **`analysis_years`** — optional cohort label suffix on the title (from §03).
- **Module** — `load("08_report_settings")` → `config/08_report_settings.py` → `report`.


In [ ]:
REPORT_TITLE = "Non-invasive radiological biomarkers of meningiomas as a prognostic tool for predicting tumor histological grade"
REPORT_AUTHOR = "Doc Arturs Balodis, Sigita Zālīte, Roberts Tumeļkāns, Valērija Aksjonova, Elizabete Stankeviča, Andris Zaguzovs"
REPORT_PATH = OUTPUT_ROOT / "report" / "report.html"

In [ ]:
_c08 = load("08_report_settings")
_c08.run_report(
    output_root=OUTPUT_ROOT,
    report_title=REPORT_TITLE,
    report_author=REPORT_AUTHOR,
    report_path=REPORT_PATH,
    analysis_years=ANALYSIS_YEARS,
    eda_targets=EDA_TARGETS,
)
_c08.print_output_summary(OUTPUT_ROOT)